# YOLOv8 Facial Expression Detection — Full Pipeline (Notebook)
**Classes:** `happy`, `sad`, `angry`, `surprised`, `scared`

Notebook ini menggabungkan seluruh pipeline dari proyekmu:
1. Setup lingkungan (install package & cek GPU)
2. Ekstraksi frame dari video
3. Deteksi wajah YOLO + auto-label by timestamp (crop 224×224)
4. Split dataset train/val
5. Training klasifikasi ekspresi (YOLOv8-CLS)
6. Inference realtime via kamera


In [ ]:
# === Setup lingkungan (jalankan saat pertama) ===
# Untuk Windows + RTX 3050 (CUDA 11.8). Untuk CPU-only, hapus --index-url di baris torch.
!pip install -q ultralytics opencv-python
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

import torch, cv2, os, sys
from ultralytics import YOLO
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("OpenCV:", cv2.__version__)


## 1) Ekstraksi Frame dari Video

In [ ]:
import cv2, os, sys



video = sys.argv[1]                          
outdir = sys.argv[2]                        
fps_keep = float(sys.argv[3]) if len(sys.argv) > 3 else 5.0

os.makedirs(outdir, exist_ok=True)

cap = cv2.VideoCapture(video)
if not cap.isOpened():
    raise RuntimeError(f"Tidak bisa membuka video: {video}")

src_fps = cap.get(cv2.CAP_PROP_FPS) or 30
step = max(int(round(src_fps / fps_keep)), 1)

i = 0
saved = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if i % step == 0:
        cv2.imwrite(os.path.join(outdir, f"f_{saved:06d}.jpg"), frame)
        saved += 1
    i += 1

cap.release()
print(f"FPS sumber ~{src_fps:.2f}, simpan tiap {step} frame -> {saved} gambar di {outdir}")


In [ ]:
# Contoh pemakaian (ubah path sesuai videomu)
# python scripts/extract_frames.py videos/raw/userA.mp4 dataset/images/userA 5
# Jika menjalankan dari notebook, kamu bisa gunakan !python
# !python scripts/extract_frames.py videos/raw/userA.mp4 dataset/images/userA 5


## 2) Deteksi Wajah (YOLO-only) + Auto-label by Timestamp (Crop 224×224)

In [ ]:
import os, sys, glob, cv2
from ultralytics import YOLO


src_dir = sys.argv[1]
out_root = sys.argv[2]
fps_keep = float(sys.argv[3])
conf_th = float(sys.argv[4]) if len(sys.argv) > 4 else 0.15 

face_model_path = "yolov8n-face.pt"
if not os.path.isfile(face_model_path):
    raise FileNotFoundError(
        "yolov8n-face.pt tidak ditemukan. Unduh dulu dan letakkan di root project."
    )

classes = ["happy","sad","angry","surprised","scared"]
segment_seconds = 10.0
tmp_root = os.path.join(out_root, "_tmp")
for c in classes:
    os.makedirs(os.path.join(tmp_root, c), exist_ok=True)

face_model = YOLO(face_model_path)  # 100% YOLO
images = sorted(glob.glob(os.path.join(src_dir, "*.jpg")))
saved_per_class = {k:0 for k in classes}

def idx_to_class(idx: int) -> str:
    t = idx / fps_keep
    seg = int(t // segment_seconds)
    if seg < 0: seg = 0
    if seg > 4: seg = 4
    return classes[seg]

for i, p in enumerate(images):
    img = cv2.imread(p)
    if img is None:
        continue

    res = face_model.predict(source=img, imgsz=640, conf=conf_th, verbose=False)
    boxes = []
    for r in res:
        if r.boxes is None: 
            continue
        for b in r.boxes:
            x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
            h, w = img.shape[:2]
            dx = int(0.10 * (x2-x1)); dy = int(0.10 * (y2-y1))
            x1 = max(0, x1 - dx); y1 = max(0, y1 - dy)
            x2 = min(w, x2 + dx); y2 = min(h, y2 + dy)
            if x2 > x1 and y2 > y1:
                boxes.append((x1,y1,x2,y2))

    if not boxes:
        continue

    label = idx_to_class(i)
    out_dir = os.path.join(tmp_root, label)

    for k, (x1,y1,x2,y2) in enumerate(boxes):
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        crop = cv2.resize(crop, (224,224), interpolation=cv2.INTER_LINEAR)
        fname = f"{os.path.basename(src_dir)}_{i:06d}_{k}.jpg"
        cv2.imwrite(os.path.join(out_dir, fname), crop)
        saved_per_class[label] += 1

print("Selesai crop (YOLO-only) ke:", tmp_root)
for c in classes:
    print(f"{c}: {saved_per_class[c]}")


In [ ]:
# Contoh pemakaian (jalankan setelah extract frames):
# !python scripts/auto_crop_faces.py dataset/images/userA dataset/faces_cls_dataset 5 0.15
# !python scripts/auto_crop_faces.py dataset/images/userB dataset/faces_cls_dataset 5 0.15
# ... dst untuk semua video


## 3) Split Dataset (train/val = 80/20)

In [ ]:
import os, random, shutil, glob, sys

root = sys.argv[1]                    
val_ratio = float(sys.argv[2]) if len(sys.argv) > 2 else 0.2

tmp = os.path.join(root, "_tmp")      
train_dir = os.path.join(root, "train")
val_dir = os.path.join(root, "val")

assert os.path.isdir(tmp), "Run auto_crop_faces.py dulu."

for c in os.listdir(tmp):
    src = os.path.join(tmp, c)
    if not os.path.isdir(src):
        continue

    imgs = glob.glob(os.path.join(src, "*.jpg"))
    random.shuffle(imgs)
    n_val = int(len(imgs) * val_ratio)

    os.makedirs(os.path.join(train_dir, c), exist_ok=True)
    os.makedirs(os.path.join(val_dir, c), exist_ok=True)

    for i, p in enumerate(imgs):
        dst_root = val_dir if i < n_val else train_dir
        shutil.copy(p, os.path.join(dst_root, c, os.path.basename(p)))

print("Split done.")


In [ ]:
# Contoh pemakaian:
# !python scripts/auto_split.py dataset/faces_cls_dataset 0.2


## 4) Training Klasifikasi (YOLOv8-CLS)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")

# Set aman untuk Windows: nonaktifkan multiprocessing (workers=0)
model.train(
    data="dataset/faces_cls_dataset",  # folder berisi train/ dan val/ (5 kelas)
    epochs=60,
    imgsz=224,
    batch=8,      # naikkan ke 16 kalau lancar
    device=0,     # GPU RTX 3050
    workers=0,    # kunci: hindari error shared memory 1455
    patience=10
)
print("Training selesai.")

**Catatan:**
- Jika kehabisan VRAM/shared memory di Windows, kecilkan `batch` dan pastikan `workers=0`.
- Model terbaik akan tersimpan di `runs/classify/train*/weights/best.pt`.


## 5) Realtime Inference via Webcam (YOLO Face + Classifier)

In [ ]:
# scripts/realtime.py
import sys, time, os
import cv2
import torch
from ultralytics import YOLO

# ---- Argumen ----
# python scripts/realtime.py <path_ke_best_classifier.pt> [cam_index] [det_conf] [cls_imgsz]
# contoh:
# python scripts/realtime.py runs/classify/train2/weights/best.pt 0 0.25 224
cls_w = sys.argv[1]
cam_index = int(sys.argv[2]) if len(sys.argv) > 2 else 0
DET_CONF = float(sys.argv[3]) if len(sys.argv) > 3 else 0.25
CLS_IMGSZ = int(sys.argv[4]) if len(sys.argv) > 4 else 224

# ---- Cek weight deteksi wajah YOLO ----
FACE_W = "yolov8n-face.pt"
if not os.path.isfile(FACE_W):
    raise FileNotFoundError(
        "File 'yolov8n-face.pt' tidak ditemukan di root project. "
        "Unduh dulu (sudah kamu punya ~6.3MB)."
    )

# ---- Load model ----
device = 0 if torch.cuda.is_available() else None
face_model = YOLO(FACE_W)
cls_model  = YOLO(cls_w)
names = cls_model.names  # idx -> label

# ---- Kamera ----
cap = cv2.VideoCapture(cam_index)
# Resolusi bisa diperkecil untuk FPS lebih tinggi
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  960)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 540)

prev_t = time.time()

def expand_box(x1,y1,x2,y2,w,h,ratio=0.10):
    dx = int(ratio*(x2-x1)); dy = int(ratio*(y2-y1))
    x1 = max(0, x1-dx); y1 = max(0, y1-dy)
    x2 = min(w, x2+dx); y2 = min(h, y2+dy)
    return x1,y1,x2,y2

while True:
    ok, frame = cap.read()
    if not ok:
        break
    H, W = frame.shape[:2]

    # --- DETEKSI WAJAH (YOLO) ---
    det = face_model.predict(
        source=frame, imgsz=640, conf=DET_CONF, verbose=False, device=device
    )
    boxes = []
    for r in det:
        if r.boxes is None: continue
        for b in r.boxes:
            x1,y1,x2,y2 = map(int, b.xyxy[0].tolist())
            x1,y1,x2,y2 = expand_box(x1,y1,x2,y2,W,H,ratio=0.10)
            if x2>x1 and y2>y1:
                boxes.append((x1,y1,x2,y2))

    labels = []
    if boxes:
        # crop -> resize 224x224
        crops = []
        for (x1,y1,x2,y2) in boxes:
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                crops.append(None)
            else:
                crops.append(cv2.resize(crop, (CLS_IMGSZ, CLS_IMGSZ), interpolation=cv2.INTER_LINEAR))

        # batch klasifikasi
        batch = [c for c in crops if c is not None]
        preds = []
        if batch:
            res = cls_model.predict(source=batch, imgsz=CLS_IMGSZ, verbose=False, device=device)
            for rr in res:
                p = rr.probs
                idx = int(p.top1)
                conf = float(p.top1conf)
                preds.append((names[idx], conf))

        # map kembali ke setiap box
        j = 0
        for c in crops:
            if c is None:
                labels.append(("unknown", 0.0))
            else:
                labels.append(preds[j]); j+=1

    # --- Gambar hasil ---
    for (x1,y1,x2,y2),(lab,conf) in zip(boxes, labels):
        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(frame, f"{lab} {conf:.2f}", (x1, max(0,y1-8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    # FPS
    now = time.time()
    fps = 1.0/(now-prev_t) if now>prev_t else 0.0
    prev_t = now
    cv2.putText(frame, f"FPS: {fps:.1f}", (10,30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

    cv2.imshow("Realtime Expression (YOLOv8)", frame)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
# Contoh pemakaian (ubah path weight sesuai hasil training):
# !python scripts/realtime.py runs/classify/train2/weights/best.pt 0
# Atau ganti threshold dan ukuran input klasifikasi:
# !python scripts/realtime.py runs/classify/train2/weights/best.pt 0 0.25 224
